### Determinando parâmetros com matrizes ABCD


A seguir teremos como objetivo identificar quais são as razões $Z_{R_P}/L$ e $Z_{R_C}/L$ dado o nosso setup experimental. Ou seja, considerando a posição dos elementos ópticos e suas distâncias iremos encontrar os valores citados.

As contas anteriores faziam considerações pouco realistas em relação a nosso sistema, como por exemplo dizer que a lente estava na cintura do feixe gaussiano.

Para corrigir os possíveis desvios devido a esses erros vamos utilizar da matriz ABCD para calcular o novo comprimento de Rayleigh após a propagação do feixe por uma série de sistemas ópticos. Vamos começar com o sistema mais simples, em que o feixe propaga até a lente de bombeio $f_p$ e é focalizada no cristal SPDC. Vamos, a princípio considerar que o cristal está na cintura ($W_0$) do feixe após ele ter passado pela lente (temos que garantir isso experimentalmente para que os resultados do Schwaller funcionem).

Abaixo temos uma representação do sistema: cintura do feixe $W_0$, propaga distância $d$ até a lente de bombeio $f_P$, atravessa a lente e propaga uma distância $d$ até o cristal SPDC. Utilizaremos a lei ABCD:
$$
q_2 = \frac{Aq_1 + B}{Cq_1 + D}
$$
e considerando que o parâmetro complexo $q(z) = z + i z_R$.

Considerando $z = 0$ a cintura do feixe gaussiano, $q$ após a propagação por uma distância $d$ até a lente nos dá:

$q'_1 = q_1 + d $, sendo $A = 1$, $B = d$, $C = 0$ e $D = 1$

Agora após passar por uma lente ($A = 1$, $B = 0$, $C = -1/f$ e $D = 1$):

$$
q_P = \frac{q'_1}{(-1/f)q'_1 + 1}
$$

Como queremos $z_{R_P}$ basta pegarmos a parte imaginária de $q_P$. Além disso, podemos definir a posição da nova cintura após a passagem pela lente pegando a parte real de $q_P$. Logo:

$$
z_{R_P} = \cal{Im} (q_P)
$$
$$
z_{0_P} = \cal{Re} (q_P)
$$

Assim poderemos determinar a razão $z_{R_P}/L = N_P$ e a distância da nova cintura da lente (poderemos comparar isso com o que encontramos através da medição com o perfilomêtro).

Vamos usar as definições acima para evitar possíveis erros algébricos (sugestão do Chat), mas a seguir está a conta que é feita diretamente.

-------------



Substituindo $q'_1$ fazendo as contas e pegando apenas a parte imaginária (comprimento de Rayleigh) obtemos ao final:

$$
z_{R_P} = \frac{z_R f^2}{(f- d)^2 + z^2_R}
$$

Em que $d$ é a distância da cintura do feixe até a lente, $f$ a distância focal e $z_R$ o comprimento de Rayleigh original.

Como queremos determinar qual a razão $z_{R_P}/ L = N_P$ para o nosso sistema basta substituir:

$$
N_P = \frac{z_R f^2}{L[(f- d)^2 + z^2_R]}
$$

In [1]:
import numpy as np


def calculate_N_ABCD(w0, wavelength, M2, d, f, L):

    # Comprimento de Rayleigh efetivo
    zR = np.pi * w0**2 / (M2 * wavelength)

    # Parâmetro complexo na cintura inicial
    q0 = 1j * zR

    # Propagação até a lente
    q1 = q0 + d

    # Transformação pela lente
    q2 = q1 / (1 - q1/f)

    # Novo comprimento de Rayleigh
    zR_new = np.imag(q2)

    # Nova cintura
    w0_new = np.sqrt(
        M2 * wavelength * zR_new / np.pi
    )

    # Posição da cintura em relação à lente
    waist_position = -np.real(q2)

    return zR_new/L, zR_new, w0_new, waist_position

f = 250e-3
L = 2e-3
wavelength = 325e-9

w0x = 385.77e-6
w0y = 473.21e-6

d_X = 149e-2 - 387.991e-3
d_Y = 149e-2 - 421.761e-3

M2_X = 1.3867555844911676
M2_Y = 1.5168027465273957

for axis, w0, d, M2 in [
    ("X", w0x, d_X, M2_X),
    ("Y", w0y, d_Y, M2_Y)
]:
    N, zR_new, w0_new, waist_pos = calculate_N_ABCD(
        w0, wavelength, M2, d, f, L
    )

    print(f"Eixo {axis}:")
    print(f"  zR'/L = {N:.2f}")
    print(f"  zR' = {zR_new*1e3:.2f} mm")
    print(f"  w0 = {w0_new*1e6:.2f} mu m")
    print(f"  Posição da cintura = {waist_pos*1e3:.2f} mm")

Eixo X:
  zR'/L = 17.99
  zR' = 35.98 mm
  w0 = 71.84 mu m
  Posição da cintura = 279.55 mm
Eixo Y:
  zR'/L = 16.48
  zR' = 32.96 mm
  w0 = 71.92 mu m
  Posição da cintura = 268.90 mm


| Parâmetro | Eixo X | Eixo Y |
|---|---:|---:|
|$z_R'/L$ | 17.99 | 16.48 |
| $z_R'$ (mm) | 35.98 | 32.9 |
| $w_0$ (µm) | 71.84 | 71.92 |
| Posição da nova cintura (mm) | 279.55 | 268.90 |

Agora, devemos fazer a mesma coisa para o feixe contra-propagante. Para isso vamos fazer uma concatenação de matrizes M para simplificar o processo. Afinal até saindo da fibra o feixe contra-propagante se propaga $d_0 = 11$mm, passa por uma lente asférica de $11mm$, propaga $d_1 = 200$mm, entra em um telescópio de $f_2 = 200$mm e $f_1 = 250$mm (propagando $450$mm entre as lentes), se propaga por $325$mm e por fim uma lente de coleta de $f_C = 75$mm e propaga até o cristal (a princípio estaria na cintura do feixe contra-propagante).

O código abaixo contém duas falhas: assumir que a cintura do feixe está na fibra e que o $M^2 = 1.$.

In [2]:
import numpy as np

# ============================================================
# FUNÇÕES: MATRIZES ABCD
# ============================================================

def propagation(d):
    """Matriz de propagação livre por uma distância d [m]."""
    return np.array([
        [1.0, d],
        [0.0, 1.0]
    ])


def lens(f):
    """Matriz de lente fina de distância focal f [m]."""
    return np.array([
        [1.0, 0.0],
        [-1.0/f, 1.0]
    ])


def transform_q(M, q):
    """Transformação ABCD do parâmetro complexo q."""
    A, B = M[0]
    C, D = M[1]

    return (A*q + B)/(C*q + D)


# ============================================================
# PARÂMETROS INICIAIS DO FEIXE CONTRA-PROPAGANTE
# ============================================================

wavelength = 650e-9       # Comprimento de onda [m]
MFD = 4.5e-6              # Mode Field Diameter da fibra [m]
w0 = MFD / 2              # Raio da cintura na fibra [m]

M2 = 1.0                  # Fator de qualidade do feixe
L = 2e-3                  # Comprimento do cristal [m]


# ============================================================
# PARÂMETROS DO SISTEMA ÓPTICO
# ============================================================

# Fibra -> lente asférica
d0 = 11e-3
fa = 11e-3

# Lente asférica -> telescópio
d1 = 211e-3

# Telescópio (primeira lente)
f2 = 200e-3

# Distância entre as lentes do telescópio
d2 = 450e-3

# Telescópio (segunda lente)
f1 = 250e-3

# Telescópio -> lente de coleta
d3 = 325e-3

# Lente de coleta
fC = 75e-3


# ============================================================
# PARÂMETRO COMPLEXO INICIAL
# ============================================================

# Comprimento de Rayleigh efetivo na fibra
zR_initial = np.pi * w0**2 / (M2 * wavelength)

# Consideramos a cintura na saída da fibra
q0 = 1j * zR_initial


# ============================================================
# MATRIZ ABCD TOTAL
# ============================================================

# A ordem das operações é da direita para a esquerda.

M_total = (
    lens(fC)
    @ propagation(d3)
    @ lens(f1)
    @ propagation(d2)
    @ lens(f2)
    @ propagation(d1)
    @ lens(fa)
    @ propagation(d0)
)


# Parâmetro imediatamente após a lente de coleta
q_final = transform_q(M_total, q0)

# Comprimento de Rayleigh após a lente de coleta
zR_C = np.imag(q_final)

# Raio da nova cintura
w0_C = np.sqrt(M2 * wavelength * zR_C / np.pi)

# Razão de interesse
N_C = zR_C / L


print("Sistema de coleta - Matrizes ABCD")

print(f"\nComprimento de onda: {wavelength*1e9:.1f} nm")
print(f"MFD da fibra: {MFD*1e6:.2f} µm")
print(f"M²: {M2:.2f}")
print(f"Comprimento do cristal: {L*1e3:.2f} mm")

print("\n--- FEIXE INICIAL ---")
print(f"Cintura inicial: {w0*1e6:.3f} µm")
print(f"Rayleigh inicial: {zR_initial*1e6:.3f} µm")

print("\n--- FEIXE APÓS A LENTE DE COLETA ---")
print(f"Rayleigh final: {zR_C*1e3:.4f} mm")
print(f"Cintura final: {w0_C*1e6:.4f} µm")

print("\n--- RAZÃO DE INTERESSE ---")
print(f"zR_C / L = {N_C:.6f}")

print("\n--- VERIFICAÇÃO DA MATRIZ ---")
print(f"Determinante ABCD: {np.linalg.det(M_total):.6f}")

print("=" * 50)


Sistema de coleta - Matrizes ABCD

Comprimento de onda: 650.0 nm
MFD da fibra: 4.50 µm
M²: 1.00
Comprimento do cristal: 2.00 mm

--- FEIXE INICIAL ---
Cintura inicial: 2.250 µm
Rayleigh inicial: 24.468 µm

--- FEIXE APÓS A LENTE DE COLETA ---
Rayleigh final: 0.7280 mm
Cintura final: 12.2727 µm

--- RAZÃO DE INTERESSE ---
zR_C / L = 0.363989

--- VERIFICAÇÃO DA MATRIZ ---
Determinante ABCD: 1.000000
